In [8]:
import os
import numpy as np
import soundfile as sf
import librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

Augmentation

In [2]:
def augment_pitch_shift(y, sr, n_semitones):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_semitones)

def augment_time_stretch(y, rate):
    return librosa.effects.time_stretch(y, rate=rate)

def augment_add_noise(y, noise_factor=0.005):
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

np array from librosa.load, every sr ms get the 13 mfcc using librosa.feature.mfcc, get the mean and std across all the columns in a single row using np.mean and np.std where axis=1 stands for columns, get the duration of the cry, get the energy or average "loadness" from the cry (rms) or root mean square energy. f0 is the fundamental or lowest frequency, zcr or (zero-crossing rate) measures how "piercing" a cry is by measuring how often the sound wave transitions from positive energy to negative energy through the 0-line axis.

In [ ]:
"""def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    duration = librosa.get_duration(y=y, sr=sr)
    
    rms = librosa.feature.rms(y=y)
    rms_mean = np.mean(rms)

    f0, voiced_flag, voiced_probs = librosa.pyin(
        y, fmin=librosa.note_to_hz('C4'), fmax=librosa.note_to_hz('C5')
    )
    f0_clean = f0[~np.isnan(f0)]
    f0_mean = np.mean(f0_clean) if len(f0_clean) > 0 else 450.0

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)

    feature_vector = np.hstack([mfcc_mean, mfcc_std, duration, rms_mean, f0_mean, zcr_mean])
    return feature_vector"""

In [3]:
def extract_features(file_path_or_array):
    """
    Extremely fast feature extractor. Drops heavy pyin loop 
    and replaces it with vector-optimized tracking math.
    """
    try:
        # Accept either a string path or a pre-loaded numpy array from augmentation
        if isinstance(file_path_or_array, str):
            y, sr = librosa.load(file_path_or_array, sr=22050)
        else:
            y, sr = file_path_or_array, 22050
            
        # 1. MFCCs (Mean and Std Dev) - Fast C-bound code
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        # 2. Duration
        duration = librosa.get_duration(y=y, sr=sr)
        
        # 3. Intensity / RMS (Root Mean Square energy)
        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)
        
        # 4. FAST Pitch tracking (Switched from pyin to standard vector yin)
        # Bounds: C4 (261Hz) to C5 (523Hz) matching infant cries
        try:
            f0 = librosa.yin(y, fmin=261, fmax=523, sr=sr)
            f0_clean = f0[~np.isnan(f0)]
            f0_mean = np.mean(f0_clean) if len(f0_clean) > 0 else 450.0
        except:
            f0_mean = 450.0 # Default fallback fallback if sound clip is too short
        
        # 5. Zero-Crossing Rate (ZCR)
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        
        # Combine all features into flat row
        return np.hstack([mfcc_mean, mfcc_std, duration, rms_mean, f0_mean, zcr_mean])
        
    except Exception as e:
        return None

data uploading and spliting

In [ ]:
"""def load_dataset_and_split(data_dir):
    categories = ['hungry', 'tired', 'discomfort']
    file_paths = []
    labels = []

    for category in categories:
        folder = os.path.join(data_dir, category)
        if not os.path.exists(folder):
            continue
        for file in os.listdir(folder):
            if file.endswith('.wav'):
                file_paths.append(os.path.join(folder, file))
                labels.append(category)

    X_train_paths, X_test_paths, y_train_raw, y_test_raw = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    return X_train_paths, X_test_paths, y_train_raw, y_test_raw"""


In [4]:
def load_dataset_and_split(data_dir):
    categories = ['hungry', 'tired', 'discomfort']
    file_paths = []
    labels = []
    
    for category in categories:
        folder = os.path.join(data_dir, category)
        if not os.path.exists(folder):
            continue
        for file in os.listdir(folder):
            if file.endswith('.wav'):
                file_paths.append(os.path.join(folder, file))
                labels.append(category)
                
    X_train_paths, X_test_paths, y_train_raw, y_test_raw = train_test_split(
        file_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    return X_train_paths, X_test_paths, y_train_raw, y_test_raw

In [ ]:
"""def process_and_augment_training_data(X_train_paths, y_train_raw):
    """Dynamically balances categories so minority groups match the majority group."""
    X_train = []
    y_train = []
    
    # 1. Count how many samples we have per class in our training paths
    from collections import Counter
    counts = Counter(y_train_raw)
    max_count = counts['hungry']  # This is our target number (the largest class)
    
    # 2. Extract features and balance data on the fly
    for path, label in zip(X_train_paths, y_train_raw):
        base_features = extract_features(path)
        if base_features is not None:
            X_train.append(base_features)
            y_train.append(label)
            
    # 3. Targeted loop augmentation to balance minority classes perfectly
    for category in ['tired', 'discomfort']:
        category_paths = [p for p, l in zip(X_train_paths, y_train_raw) if l == category]
        if not category_paths:
            continue
            
        current_count = len(category_paths)
        needed_clones = max_count - current_count
        
        # Clone until this category matches the 'hungry' file count
        clone_idx = 0
        while needed_clones > 0 and len(category_paths) > 0:
            path = category_paths[clone_idx % len(category_paths)]
            try:
                y, sr = librosa.load(path, sr=22050)
                
                # Apply varied random combinations of pitch and time stretch shifts
                random_pitch = np.random.uniform(-1.8, 1.8)
                random_stretch = np.random.uniform(0.9, 1.1)
                
                augmented_wave = librosa.effects.pitch_shift(y, sr=sr, n_steps=random_pitch)
                augmented_wave = librosa.effects.time_stretch(augmented_wave, rate=random_stretch)
                
                feat = extract_features(augmented_wave)
                if feat is not None:
                    X_train.append(feat)
                    y_train.append(category)
                    needed_clones -= 1
            except:
                pass
            clone_idx += 1
                    
    return np.array(X_train), np.array(y_train)"""

In [5]:
def process_and_augment_training_data(X_train_paths, y_train_raw):
    """Upgraded fast balancing algorithm utilizing vectorized preprocessing."""
    X_train = []
    y_train = []
    
    counts = Counter(y_train_raw)
    max_count = counts['hungry']  # Target balance threshold
    
    # Track paths by category to pull items quickly
    paths_by_cat = {cat: [p for p, l in zip(X_train_paths, y_train_raw) if l == cat] for cat in counts.keys()}
    
    # Step A: Parse base samples
    for path, label in zip(X_train_paths, y_train_raw):
        feat = extract_features(path)
        if feat is not None:
            X_train.append(feat)
            y_train.append(label)
    print("⚡ Base data processed. Generating balancing clones rapidly...")
        
    # Step B: Safe structural replication loop
    for category in ['tired', 'discomfort']:
        category_paths = paths_by_cat.get(category, [])
        if not category_paths:
            continue
            
        current_count = len(category_paths)
        needed_clones = max_count - current_count
        
        clone_idx = 0
        while needed_clones > 0 and len(category_paths) > 0:
            path = category_paths[clone_idx % len(category_paths)]
            try:
                # Load audio once per clone attempt
                y, sr = librosa.load(path, sr=22050)     
                # Apply varied random pitch & time scaling configurations
                random_pitch = np.random.uniform(-1.5, 1.5)
                random_stretch = np.random.uniform(0.92, 1.08)
                
                aug_y = augment_pitch_shift(y, sr, random_pitch)
                aug_y = augment_time_stretch(aug_y, rate=random_stretch)
                
                # Pass array directly to feature extractor to prevent slow disk writes
                feat = extract_features(aug_y)
                if feat is not None:
                    X_train.append(feat)
                    y_train.append(category)
                    needed_clones -= 1
            except:
                pass
            clone_idx += 1
                    
    return np.array(X_train), np.array(y_train)

In [ ]:
"""if __name__ == "__main__":
    DATASET_DIRECTORY = "./dataset"
    
    print("Step 1: Splitting dataset into clean train/test blocks...")
    X_tr_paths, X_te_paths, y_tr_raw, y_te_raw = load_dataset_and_split(DATASET_DIRECTORY)
    
    print("Step 2: Processing and augmenting Training Data...")
    X_train, y_train = process_and_augment_training_data(X_tr_paths, y_tr_raw)
    
    print("Step 3: Processing Unaugmented Test Data...")
    X_test = []
    y_test = []
    for path, label in zip(X_te_paths, y_te_raw):
        feat = extract_features(path)
        if feat is not None:
            X_test.append(feat)
            y_test.append(label)
    X_test, y_test = np.array(X_test), np.array(y_test)
    
    print("Step 4: Training Random Forest Classifier...")
    model = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    
    print("\n Evaluation Summary:")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))"""

Step 1: Splitting dataset into clean train/test blocks...
Step 2: Processing and augmenting Training Data...


/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/numba/np/ufunc/dufunc.py:303: RuntimeWarning: invalid value encountered in cast
  return super().__call__(*args, **kws)


In [7]:
from collections import Counter

if __name__ == "__main__":
    DATASET_DIRECTORY = "./dataset" 
    
    print("⏳ Step 1: Isolating train/test data splits...")
    X_tr_paths, X_te_paths, y_tr_raw, y_te_raw = load_dataset_and_split(DATASET_DIRECTORY)
    
    print("⏳ Step 2: Processing and cloning Training Data (This should take < 45 seconds now)...")
    X_train, y_train = process_and_augment_training_data(X_tr_paths, y_tr_raw)
    
    print("⏳ Step 3: Extracting clean Test Matrix values...")
    X_test = [extract_features(p) for p in X_te_paths]
    X_test = np.array([f for f in X_test if f is not None])
    y_test = np.array([l for p, l in zip(X_te_paths, y_te_raw) if extract_features(p) is not None])
    
    print("⏳ Step 4: Training Random Forest Engine...")
    model = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    print("\n📊 Balanced Metric Report:")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    print("📋 Confusion Matrix Output:")
    print(confusion_matrix(y_test, y_pred))
    
    with open('cry_classifier_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    print("\n💾 Success! Clean model saved to 'cry_classifier_model.pkl'.")

⏳ Step 1: Isolating train/test data splits...
⏳ Step 2: Processing and cloning Training Data (This should take < 45 seconds now)...


⚡ Base data processed. Generating balancing clones rapidly...


/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/numba/np/ufunc/dufunc.py:303: RuntimeWarning: invalid value encountered in cast
  return super().__call__(*args, **kws)


⏳ Step 3: Extracting clean Test Matrix values...
⏳ Step 4: Training Random Forest Engine...

📊 Balanced Metric Report:
              precision    recall  f1-score   support

  discomfort       0.00      0.00      0.00        10
      hungry       0.86      0.94      0.89        77
       tired       0.00      0.00      0.00         5

    accuracy                           0.78        92
   macro avg       0.29      0.31      0.30        92
weighted avg       0.72      0.78      0.75        92

📋 Confusion Matrix Output:
[[ 0  9  1]
 [ 3 72  2]
 [ 2  3  0]]

💾 Success! Clean model saved to 'cry_classifier_model.pkl'.


In [ ]:
with open('cry_classifier_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("\n💾 Model successfully saved as 'cry_classifier_model.pkl'!")

In [ ]:
print(classification_report(y_test, y_pred))
    
print("📋 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))